# Time Series Forecasting: Exercises

This notebook accompanies the `06_time_series_forecasting` module. You will
explore synthetic Indonesian outlet sales, compare classical forecasts, fit an
AR model, build leakage-free lag features and run a walk-forward backtest.

## Learning objectives

- Recognise trend, weekly seasonality, payday (`gajian`) spikes and holidays.
- Split a time series chronologically, without shuffling.
- Compare naive, seasonal-naive, moving-average, SES and Holt-Winters models.
- Estimate an `AR(p)` with least squares and forecast recursively.
- Build lag/rolling features without look-ahead leakage.
- Evaluate forecasts with MAE, RMSE, MAPE and SMAPE.
- Validate with rolling-origin backtesting.

Every exercise is followed by a worked solution. Run the notebook from top to
bottom; the setup cell works from this directory or from the repository root.

## 0. Setup

Import the from-scratch implementation and set a reproducible random state.

In [ ]:
import os
import sys

import numpy as np

MODULE_DIR = os.path.abspath(".")
if not os.path.exists(os.path.join(MODULE_DIR, "implementation.py")):
    MODULE_DIR = os.path.abspath("04_advanced_topics/06_time_series_forecasting")
sys.path.insert(0, MODULE_DIR)

from implementation import (
    ARModel,
    HoltWinters,
    LagRegression,
    SimpleExpSmoothing,
    generate_outlet_sales,
    mae,
    mape,
    make_lag_features,
    moving_average_forecast,
    naive_forecast,
    rmse,
    rolling_origin_backtest,
    seasonal_naive_forecast,
    smape,
    time_train_test_split,
)

np.set_printoptions(precision=2, suppress=True)
RANDOM_STATE = 42
print("Module directory:", MODULE_DIR)

## Exercise 1: Explore the synthetic sales

Generate one year of daily sales. Then answer:

1. What is the date range and the average sales level?
2. Which weekdays are busiest?
3. Is the payday window (25th--2nd) stronger than an ordinary mid-month window?

**Solution:** the next cell computes weekday means and compares payday with
ordinary days.

In [ ]:
dates, sales = generate_outlet_sales(n_days=365, seed=RANDOM_STATE)

print("Observations:", sales.size)
print("Date range:", dates[0], "to", dates[-1])
print("Sales mean / min / max:", round(sales.mean(), 2), sales.min(), sales.max())

epoch_days = dates.astype("datetime64[D]").astype(np.int64)
weekday = (epoch_days + 3) % 7
weekday_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
weekday_means = np.array([sales[weekday == d].mean() for d in range(7)])

for name, value in zip(weekday_names, weekday_means):
    print(f"{name}: {value:8.2f}")

month_start = dates.astype("datetime64[M]")
day_of_month = (dates - month_start).astype(np.int64) + 1
payday = np.isin(day_of_month, [25, 26, 27, 28, 29, 30, 31, 1, 2])
ordinary = np.isin(day_of_month, [10, 11, 12, 13, 14, 15, 16])
print("Payday-window mean:", round(sales[payday].mean(), 2))
print("Ordinary-day mean: ", round(sales[ordinary].mean(), 2))
assert sales[payday].mean() > sales[ordinary].mean()

## Exercise 2: Ordered train/test split

Split the series into 335 training days and 30 test days.

**Question:** why is a random shuffle invalid here? Check that the training and
test slices are contiguous, non-overlapping and in chronological order.

**Solution:** `time_train_test_split` removes the last `test_size` observations
without shuffling.

In [ ]:
horizon = 30
train, test = time_train_test_split(sales, test_size=horizon)

print("Train:", train.size, "Test:", test.size)
print("Last train value:", train[-1], "First test value:", test[0])
print("Training dates end before test dates begin by construction.")
assert train.size + test.size == sales.size
assert np.array_equal(np.concatenate([train, test]), sales)

## Exercise 3: Baseline forecasts

Forecast the 30 test days with:

- naive (last value repeated),
- seasonal naive (`season=7`),
- moving average (`window=7`).

Compute MAE and MAPE for each. Does seasonal naive beat naive on this strongly
seasonal series?

**Solution:** run the comparison and check the assertion.

In [ ]:
baselines = {
    "Naive": naive_forecast(train, horizon),
    "Seasonal naive (7)": seasonal_naive_forecast(train, horizon, season=7),
    "Moving average (7)": moving_average_forecast(train, horizon, window=7),
}

print(f"{'Model':<22} {'MAE':>10} {'MAPE %':>10}")
for name, prediction in baselines.items():
    print(f"{name:<22} {mae(test, prediction):>10.2f} {mape(test, prediction):>10.2f}")

assert mae(test, baselines["Seasonal naive (7)"]) < mae(test, baselines["Naive"])

## Exercise 4: Exponential smoothing and Holt-Winters

Fit simple exponential smoothing and additive Holt-Winters, then compare them
with the naive baseline.

**Questions:**

1. Why can SES capture the level but not the weekly pattern?
2. What do the fitted Holt-Winters `level_`, `trend_` and `seasonal_` mean?
3. Does Holt-Winters beat naive on the test window?

**Solution:** the next cell fits both models and prints the learned states.

In [ ]:
ses = SimpleExpSmoothing(alpha=0.3).fit(train)
ses_prediction = ses.forecast(horizon)

hw = HoltWinters(alpha=0.3, beta=0.05, gamma=0.3, season_length=7).fit(train)
hw_prediction = hw.forecast(horizon)

for name, prediction in [
    ("SES", ses_prediction),
    ("Holt-Winters", hw_prediction),
]:
    print(f"{name:<14} MAE={mae(test, prediction):7.2f}  MAPE={mape(test, prediction):6.2f}%")

print("SES final level:", round(ses.level_, 2))
print("Holt-Winters level:", round(hw.level_, 2))
print("Holt-Winters trend:", round(hw.trend_, 4))
print("Holt-Winters seasonal:", np.round(hw.seasonal_, 2))
assert mae(test, hw_prediction) < mae(test, baselines["Naive"])

## Exercise 5: Recover a known AR(2)

Simulate `y_t = 5 + 0.6 y_{t-1} - 0.25 y_{t-2} + noise`, fit `ARModel(2)`, and
check whether least squares recovers the intercept and coefficients.

**Solution:** with thousands of observations and modest noise, the estimates
should be close to the true values. Then inspect a recursive 10-step forecast.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n = 4000
ar_sales = np.zeros(n)
for t in range(2, n):
    ar_sales[t] = 5.0 + 0.6 * ar_sales[t - 1] - 0.25 * ar_sales[t - 2] + rng.normal(0, 0.5)

ar_model = ARModel(p=2).fit(ar_sales)
print("Estimated intercept:", round(ar_model.intercept_, 3))
print("Estimated coefficients:", np.round(ar_model.coef_, 3))
print("Recursive 10-step forecast:", np.round(ar_model.forecast(10), 2))

assert abs(ar_model.intercept_ - 5.0) < 0.15
assert np.allclose(ar_model.coef_, [0.6, -0.25], atol=0.05)

## Exercise 6: Lag features without leakage

Build features with lags 1, 2 and 7 plus rolling windows 7 and 14.

**Question:** does the first row use the target value anywhere? Prove it by
changing every observation from index 40 onward and checking that feature rows
for targets before index 40 are unchanged.

**Solution:** `make_lag_features` aligns row `i` with target `y[start+i]` and
only uses observations before that timestamp.

In [ ]:
y = np.arange(1, 81, dtype=float)
X, target, index = make_lag_features(
    y, lags=(1, 2, 7), rolling_windows=(7, 14), return_index=True
)

start = 14
print("First target index:", index[0])
print("First target value:", target[0])
print("First row:", np.round(X[0], 2))
assert target[0] == y[start]
assert X[0, 0] == y[start - 1]
assert X[0, 2] == y[start - 7]
assert X[0, 3] == np.mean(y[start - 7:start])

changed = y.copy()
changed[40:] += 10_000
X_changed, _ = make_lag_features(
    changed, lags=(1, 2, 7), rolling_windows=(7, 14)
)
safe = target < 40
assert np.allclose(X[safe], X_changed[safe])
print("No leakage: rows before the changed region are identical.")

## Exercise 7: Recursive lag regression

Fit `LagRegression` with the same lags and rolling windows used in the
scikit-learn example, then forecast the 30-day test horizon recursively.

**Question:** why can a recursive forecast drift away from the actual values
faster than a one-step-ahead model?

**Solution:** each prediction becomes an input for the next step, so errors
compound. Compare the result with the seasonal-naive baseline.

In [ ]:
lag_model = LagRegression(
    lags=(1, 2, 3, 4, 5, 6, 7, 14), rolling_windows=(7, 14)
).fit(train)
lag_prediction = lag_model.forecast(horizon)

print("Feature names:", lag_model.feature_names_)
print("LagRegression MAE:", round(mae(test, lag_prediction), 2))
print("Seasonal naive MAE:", round(mae(test, baselines["Seasonal naive (7)"]), 2))
print("First five recursive forecasts:", np.round(lag_prediction[:5], 2))
assert lag_prediction.shape == (horizon,)
assert np.all(np.isfinite(lag_prediction))

## Exercise 8: Rolling-origin backtest

Run a walk-forward backtest on the full series with a seasonal-naive model:
first training window 250 days, forecast horizon 30 days, step 30 days.

**Question:** how many complete folds fit in 365 observations?

**Solution:** origins are 250, 280 and 310, so there are three folds. Each fold
trains only on observations before its test window.

In [ ]:
folds = rolling_origin_backtest(
    sales,
    lambda history, h: seasonal_naive_forecast(history, h, season=7),
    initial=250,
    horizon=30,
    step=30,
)

print("Number of folds:", len(folds))
for fold in folds:
    print(
        f"fold {fold['fold']}: train_end={fold['train_end']}, "
        f"test=[{fold['test_start']}, {fold['test_end']}), "
        f"MAE={fold['mae']:.2f}"
    )
assert len(folds) == 3
assert all(fold["train_end"] <= fold["test_start"] for fold in folds)

## Exercise 9: Challenge -- choose a production forecaster

Using the test-window predictions already computed, rank all models by MAE and
MAPE. Then discuss:

1. Which model wins this particular 30-day window, and is that enough evidence?
2. How would you select hyperparameters without touching the test window?
3. What extra features could capture payday and holiday effects more explicitly?
4. Why is SMAPE useful when some low-sales days are near zero?

**Solution starter:** the next cell builds the ranking. Extend it with your own
models and with the scikit-learn models from `sklearn_example.py`.

In [ ]:
candidates = {
    "Naive": baselines["Naive"],
    "Seasonal naive": baselines["Seasonal naive (7)"],
    "Moving average": baselines["Moving average (7)"],
    "SES": ses_prediction,
    "Holt-Winters": hw_prediction,
    "LagRegression": lag_prediction,
}

ranking = sorted(
    (
        (name, mae(test, prediction), mape(test, prediction))
        for name, prediction in candidates.items()
    ),
    key=lambda row: row[1],
)

print(f"{'Rank':<6} {'Model':<20} {'MAE':>10} {'MAPE %':>10}")
for rank, (name, error, percentage) in enumerate(ranking, start=1):
    print(f"{rank:<6} {name:<20} {error:>10.2f} {percentage:>10.2f}")

print()
print("Best on this window:", ranking[0][0])
print("Remember: one test window is not a model-selection procedure.")